In [19]:
!pip install ninja

In [20]:
!nvidia-smi
!nvcc --version

Sun May 31 16:27:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [21]:
import torch
print(f"Pytorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available}")
print(f"Device: {torch.cuda.get_device_name(0)}")

Pytorch: 2.11.0+cu128
CUDA available: <function is_available at 0x7f8b1c5342c0>
Device: Tesla T4


In [27]:
import os
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'

!rm -rf /root/.cache/torch_extensions/

import torch
from torch.utils.cpp_extension import load_inline

cuda_source = '''
#include <torch/extension.h>

__global__ void add_one_kernel(float* x, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) x[i] = x[i] + 1.0f;
}

torch::Tensor add_one(torch::Tensor x) {
    int n = x.numel();
    int threads = 256;
    int blocks = (n + threads - 1) / threads;
    add_one_kernel<<<blocks, threads>>>(x.data_ptr<float>(), n);
    return x;
}
'''

cpp_source = "torch::Tensor add_one(torch::Tensor x);"

mod = load_inline(
    name="add_one_mod",
    cpp_sources=cpp_source,
    cuda_sources=cuda_source,
    functions=["add_one"],
    verbose=True,
    with_cuda=True,
)

x = torch.zeros(10, device='cuda')
print("Before:", x)
mod.add_one(x)
print("After: ", x)

Before: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], device='cuda:0')
After:  tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.], device='cuda:0')
